# Exercise 9.1: BDT for H → γγ

From *Programming in High Energy Particle Physics*, Chapter 9

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch09/ex09_1_solution.ipynb)

Exercise 9.1 BDT for H → γγ Train an XGBoost classifier to distinguish \(H\to\gamma\gamma\) signal from diphoton background. Use features: leading photon \(p_T\), subleading photon \(p_T\), diphoton mass, diphoton \(p_T\), and \(\cos\theta^*\). Optimize hyperparameters using cross-validation.

<details><summary>Hint</summary>

Use sklearn.model_selection.GridSearchCV or RandomizedSearchCV with StratifiedKFold over max_depth , learning_rate and n_estimators , scoring with "roc_auc" . Then look at the background \(m_{\gamma\gamma}\) distribution before and after a cut on the score. Because the signal is a narrow peak at 125 GeV, the classifier will learn to select events near that mass and sculpt a peak into the background. Real \(H\to\gamma\gamma\) analyses therefore keep \(m_{\gamma\gamma}\) out of the classifier, use \(p_T/m_{\gamma\gamma}\) instead of the raw photon \(p_T\), and fit the \(m_{\gamma\gamma}\) spectrum after categorizing events by the score. Compare both versions.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q numpy matplotlib xgboost scikit-learn")

**Data:** seeded synthetic diphoton features; no collision data are used. The train/test split is fixed before cross-validation.

### Step 1: Generate a reproducible diphoton toy sample

In [ ]:
import numpy as np
rng=np.random.default_rng(42); N=3000; y=np.r_[np.ones(N,dtype=int),np.zeros(N,dtype=int)]
mass=np.r_[rng.normal(125,1.8,N),rng.uniform(105,160,N)]
pt1=np.r_[rng.normal(67,12,N),rng.normal(57,15,N)]
pt2=np.r_[rng.normal(48,9,N),rng.normal(42,12,N)]
ptgg=np.r_[rng.exponential(42,N),rng.exponential(35,N)]
costh=np.r_[rng.uniform(-.8,.8,N),rng.uniform(-1,1,N)]
X=np.column_stack([pt1,pt2,mass,ptgg,costh]); assert X.shape==(2*N,5)

### Step 2: Tune and compare two BDTs

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
idx_train,idx_test=train_test_split(np.arange(len(y)),test_size=.3,stratify=y,random_state=42)
cv=StratifiedKFold(3,shuffle=True,random_state=42)
params={'max_depth':[2,3], 'learning_rate':[.08,.2], 'n_estimators':[40,80]}
scores={}
for label,A in {'mass included':X, 'mass excluded':np.column_stack([pt1/mass,pt2/mass,ptgg,costh])}.items():
    grid=GridSearchCV(XGBClassifier(tree_method='hist',n_jobs=1,random_state=42,eval_metric='logloss'),params,cv=cv,scoring='roc_auc',n_jobs=1)
    grid.fit(A[idx_train],y[idx_train]); score=grid.best_estimator_.predict_proba(A[idx_test])[:,1]
    scores[label]=score
    print(label,'CV AUC',round(grid.best_score_,3),'test AUC',round(roc_auc_score(y[idx_test],score),3),grid.best_params_)

### Step 3: Inspect background mass sculpting

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(11,4)); bkg=y[idx_test]==0
for a,(label,score) in zip(ax,scores.items()):
    cut=np.quantile(score[bkg],.85)
    bins=np.linspace(105,160,28)
    a.hist(mass[idx_test][bkg],bins=bins,histtype='step',density=True,label='before')
    a.hist(mass[idx_test][bkg & (score>cut)],bins=bins,histtype='step',density=True,label='top 15% score')
    a.set(xlabel=r'$m_{\gamma\gamma}$ [GeV]',ylabel='Normalized density [1/GeV]',title=label); a.legend()
plt.tight_layout(); plt.show()

The toy BDT that sees mass learns the narrow 125 GeV signal location and sculpts the background. The mass-excluded model uses normalized photon transverse momenta and is safer for a later mass fit. These toy AUCs are illustrative, not measured Higgs performance.